In [1]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import torch
import onnx
from model_definition import SimpleCNN

In [3]:
SCRIPT_DIR = Path().resolve()

MODEL_PATH = SCRIPT_DIR / "model.pth"
CALIB_DIR = SCRIPT_DIR / "calib"

ONNX_PATH = SCRIPT_DIR / "model.onnx"

LOG_FILE = SCRIPT_DIR / "conversion_log.txt"

In [4]:
logging.basicConfig(
    filename=LOG_FILE,
    filemode="w",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

console = logging.StreamHandler(sys.stdout)
console.setLevel(logging.INFO)
logging.getLogger().addHandler(console)

In [5]:
#Load Pytorch model
def load_pytorch_model():

    logging.info("Loading PyTorch model...")

    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Checkpoint not found:\n{MODEL_PATH}"
        )

    model = SimpleCNN()

    checkpoint = torch.load(
        MODEL_PATH,
        map_location="cpu"
    )

    model.load_state_dict(checkpoint)

    model.eval()

    logging.info("Model loaded successfully.")

    return model

In [6]:
#Validate Calibration Data
def validate_calibration_data():

    logging.info("Checking calibration directory...")

    if not CALIB_DIR.exists():
        raise FileNotFoundError(
            f"Calibration directory missing:\n{CALIB_DIR}"
        )

    files = sorted(CALIB_DIR.glob("*.npy"))

    if len(files) == 0:
        raise RuntimeError(
            "Calibration directory is empty."
        )

    calibration_samples = []

    for file in files:

        try:
            sample = np.load(file)

        except Exception as e:
            raise RuntimeError(
                f"Unable to read {file.name}\n{e}"
            )

        if sample.dtype != np.float32:
            raise TypeError(
                f"{file.name} is not float32"
            )

        if sample.shape != (1, 28, 28):
            raise ValueError(
                f"{file.name} has invalid shape {sample.shape}"
            )

        if np.isnan(sample).any():
            raise ValueError(
                f"{file.name} contains NaN"
            )

        if np.isinf(sample).any():
            raise ValueError(
                f"{file.name} contains Inf"
            )

        calibration_samples.append(sample)

    logging.info(
        f"Validated {len(calibration_samples)} calibration samples."
    )

    return calibration_samples

In [7]:
# Export to ONNX
def export_onnx(model):

    logging.info("Exporting model to ONNX...")

    dummy_input = torch.randn(
        1,
        1,
        28,
        28,
        dtype=torch.float32
    )

    torch.onnx.export(
        model,
        dummy_input,
        ONNX_PATH,
        export_params=True,
        opset_version=13,
        do_constant_folding=True,

        input_names=["input"],
        output_names=["output"],

        dynamic_axes=None
    )

    logging.info(f"ONNX model saved to:\n{ONNX_PATH}")

In [8]:
# Verify ONNX
def verify_onnx():

    logging.info("Verifying ONNX model...")

    if not ONNX_PATH.exists():
        raise FileNotFoundError(
            "ONNX export failed."
        )

    model = onnx.load(ONNX_PATH)

    onnx.checker.check_model(model)

    logging.info("ONNX verification successful.")


In [11]:
import shutil
import subprocess

In [12]:
SAVED_MODEL_DIR = SCRIPT_DIR / "saved_model"

In [19]:
def convert_to_saved_model():

    logging.info("Converting ONNX -> TensorFlow SavedModel...")

    if not ONNX_PATH.exists():
        raise FileNotFoundError(
            "model.onnx not found."
        )

    # Remove previous SavedModel
    if SAVED_MODEL_DIR.exists():
        shutil.rmtree(SAVED_MODEL_DIR)

    command = [
    "onnx2tf",
    "-i", str(ONNX_PATH),
    "-o", str(SAVED_MODEL_DIR),
    "-fdosm"
]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    logging.info(result.stdout)

    if result.returncode != 0:

        logging.error(result.stderr)

        raise RuntimeError(
            "ONNX -> TensorFlow conversion failed."
        )

    if not SAVED_MODEL_DIR.exists():
        raise RuntimeError(
            "SavedModel directory was not created."
        )

    logging.info("TensorFlow SavedModel created successfully.")

In [20]:
import tensorflow as tf

In [21]:
def verify_saved_model():

    logging.info("Loading SavedModel...")

    model = tf.saved_model.load(str(SAVED_MODEL_DIR))

    signatures = list(model.signatures.keys())

    logging.info(
        f"Available signatures: {signatures}"
    )

    print("\nSavedModel verified successfully.")
    print("Signatures:", signatures)

In [25]:
import onnx

m = onnx.load("model.onnx")
print("Opset:", m.opset_import[0].version)

Opset: 18


In [35]:
TFLITE_INT8_PATH = SCRIPT_DIR / "model_int8.tflite"

In [26]:
def representative_dataset():

    logging.info("Preparing representative dataset...")

    calibration_files = sorted(CALIB_DIR.glob("*.npy"))

    for file in calibration_files:

        sample = np.load(file).astype(np.float32)

        # Shape: (1, 28, 28) -> (1, 1, 28, 28)
        sample = np.expand_dims(sample, axis=0)

        yield [sample]

In [27]:
def convert_to_int8_tflite():

    logging.info("Converting SavedModel -> INT8 TFLite...")

    converter = tf.lite.TFLiteConverter.from_saved_model(
        str(SAVED_MODEL_DIR)
    )

    converter.optimizations = [
        tf.lite.Optimize.DEFAULT
    ]

    converter.representative_dataset = representative_dataset

    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8
    ]

    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()

    with open(TFLITE_INT8_PATH, "wb") as f:
        f.write(tflite_model)

    logging.info("INT8 model saved successfully.")

In [28]:
def verify_tflite():

    logging.info("Verifying INT8 TFLite model...")

    interpreter = tf.lite.Interpreter(
        model_path=str(TFLITE_INT8_PATH)
    )

    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()

    output_details = interpreter.get_output_details()

    print("\n===== INT8 MODEL =====")

    print("Input dtype :", input_details[0]["dtype"])
    print("Output dtype:", output_details[0]["dtype"])

    print("Input shape :", input_details[0]["shape"])

    print("Output shape:", output_details[0]["shape"])

    if input_details[0]["dtype"] == np.int8:
        print("\n✓ Input is INT8")

    else:
        print("\n✗ Input is NOT INT8")

    if output_details[0]["dtype"] == np.int8:
        print("✓ Output is INT8")

    else:
        print("✗ Output is NOT INT8")

In [33]:
def representative_dataset():

    logging.info("Preparing representative dataset...")

    calibration_files = sorted(CALIB_DIR.glob("*.npy"))

    for file in calibration_files:

        sample = np.load(file).astype(np.float32)

        # If shape is (1,28,28), remove channel dimension
        if sample.shape == (1, 28, 28):
            sample = sample.squeeze(0)

        # (28,28) -> (28,28,1)
        sample = np.expand_dims(sample, axis=-1)

        # (28,28,1) -> (1,28,28,1)
        sample = np.expand_dims(sample, axis=0)
        
        yield [sample]

sample = np.load(sorted(CALIB_DIR.glob("*.npy"))[0])

print(sample.shape)

(1, 28, 28)


In [37]:
def load_tflite_model():

    logging.info("Loading INT8 TFLite model...")

    interpreter = tf.lite.Interpreter(
        model_path=str(TFLITE_INT8_PATH)
    )

    interpreter.allocate_tensors()

    return interpreter

In [38]:
def inspect_tflite(interpreter):

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    print("\n========== TFLite Model ==========")

    print("Input Shape :", input_details[0]["shape"])
    print("Input Type  :", input_details[0]["dtype"])

    print("Output Shape:", output_details[0]["shape"])
    print("Output Type :", output_details[0]["dtype"])

    print("Input Quantization :", input_details[0]["quantization"])
    print("Output Quantization:", output_details[0]["quantization"])

In [40]:
def test_inference(interpreter):

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    sample = np.load(
        sorted(CALIB_DIR.glob("*.npy"))[0]
    ).astype(np.float32)

    # Convert to NHWC

    if sample.shape == (1,28,28):
        sample = sample.squeeze(0)

    sample = np.expand_dims(sample,-1)
    sample = np.expand_dims(sample,0)

    scale, zero = input_details[0]["quantization"]

    sample_int8 = np.round(sample / scale + zero).astype(np.int8)

    interpreter.set_tensor(
        input_details[0]["index"],
        sample_int8
    )

    interpreter.invoke()

    output = interpreter.get_tensor(
        output_details[0]["index"]
    )

    out_scale, out_zero = output_details[0]["quantization"]

    output_float = (output.astype(np.float32)-out_zero)*out_scale

    prediction = np.argmax(output_float)

    print("\nPrediction:", prediction)

In [41]:
def compare_with_pytorch(model):

    sample = np.load(
        sorted(CALIB_DIR.glob("*.npy"))[0]
    ).astype(np.float32)

    tensor = torch.from_numpy(sample).unsqueeze(0)

    with torch.no_grad():

        output = model(tensor)

        prediction = output.argmax(1).item()

    print("PyTorch Prediction :", prediction)

In [43]:
def main():


    model = load_pytorch_model()

    validate_calibration_data()

    export_onnx(model)

    verify_onnx()

    convert_to_saved_model()

    verify_saved_model()

    convert_to_int8_tflite()

    interpreter = load_tflite_model()

    inspect_tflite(interpreter)

    test_inference(interpreter)

    compare_with_pytorch(model)

    print("\nPipeline completed successfully.")


main()

Loading PyTorch model...
Model loaded successfully.
Checking calibration directory...
Validated 50 calibration samples.
Exporting model to ONNX...


W0726 21:15:09.579000 29888 torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 13).
Failed to convert the model to the target version 13 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\gouth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\onnxscript\version_converter\__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  F

C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Exception in thread Thread-31 (_readerthread):
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.

None
TensorFlow SavedModel created successfully.
Loading SavedModel...
Available signatures: ['serving_default']

SavedModel verified successfully.
Signatures: ['serving_default']
Converting SavedModel -> INT8 TFLite...
Preparing representative dataset...
INT8 model saved successfully.
Loading INT8 TFLite model...

========== TFLite Model ==========
Input Shape : [ 1 28 28  1]
Input Type  : <class 'numpy.int8'>
Output Shape: [ 1 10]
Output Type : <class 'numpy.int8'>
Input Quantization : (0.003921568859368563, -128)
Output Quantization: (0.14445307850837708, -1)

Prediction: 7
PyTorch Prediction : 7

Pipeline completed successfully.


C:\Users\gouth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
